# Day 4

## Tokenizing with code

In [ ]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")
tokens = encoding.encode("Hi my name is Ed and I like banoffee pie")
tokens


In [5]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

tokens = encoding.encode("Hi my name is Ed and I like banoffee pie")

print(tokens)
print(len(tokens))

[13347, 856, 836, 374, 3279, 323, 358, 1093, 9120, 21869, 4447]
11


In [4]:
# imported from transformer 
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("openai/gpt-oss-20b")
tokens = tokenizer.encode(
    "Hi my name is Ed and I like banoffee pie"
)

print(tokens)
print(len(tokens))

tokenizer_config.json: 0.00B [00:00, ?B/s]

e:\MK\AI projects\Agentic-core\llm_engineering\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\admin\.cache\huggingface\hub\models--openai--gpt-oss-20b. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to 

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

[12194, 922, 1308, 382, 6117, 326, 357, 1299, 9171, 26458, 5148]
11


In [6]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

13347 = Hi
856 =  my
836 =  name
374 =  is
3279 =  Ed
323 =  and
358 =  I
1093 =  like
9120 =  ban
21869 = offee
4447 =  pie


In [7]:
encoding.decode([326])

' l'

# And another topic!

### The Illusion of "memory"

Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

In [8]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
# api_key = os.getenv('OPENAI_API_KEY')
api_key = os.getenv('GROQ_API_KEY')
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook


### You should be very comfortable with what the next cell is doing!

_I'm creating a new instance of the OpenAI Python Client library, a lightweight wrapper around making HTTP calls to an endpoint for calling the GPT LLM, or other LLM providers_

In [10]:
# from openai import OpenAI
# openai = OpenAI()
from groq import Groq
groqcall = Groq()

### A message to OpenAI is a list of dicts

In [13]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"}
    ]

In [ ]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

In [15]:
response = groqcall.chat.completions.create(model="openai/gpt-oss-20b", 
                                            messages=messages,
                                            reasoning_effort="medium")
print(response.choices[0].message.content)

Hello Ed! 👋 How can I help you today?


### OK let's now ask a follow-up question

In [16]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [ ]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

In [17]:
response = groqcall.chat.completions.create(model="openai/gpt-oss-20b", 
                                            messages=messages,
                                            reasoning_effort="medium")
print(response.choices[0].message.content)

I don’t have that information. What’s your name?


### Wait, wha??

We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [18]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm MK!"},
    {"role": "assistant", "content": "Hi MK! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [ ]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

In [19]:
response = groqcall.chat.completions.create(model="openai/gpt-oss-20b", 
                                            messages=messages,
                                            reasoning_effort="medium")
print(response.choices[0].message.content)

You mentioned that your name is MK.


## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

